In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
MODE = "full"
SEED = 1337
TASKS = ("sz_air_ppo", "mz_air_ppo", "mz_air_mappo", "mz_hydro_ppo", "mz_hydro_mappo")
OUTPUT_ROOT = REPO_ROOT / "runs" / MODE / f"seed{SEED}"

In [ ]:
status_rows = []
histories = {}
for task in TASKS:
    run_dir = OUTPUT_ROOT / task
    manifest_path = run_dir / "run_manifest.json"
    latest_path = run_dir / "checkpoints" / "latest.json"
    manifest = (
        json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
    )
    latest = json.loads(latest_path.read_text(encoding="utf-8")) if latest_path.exists() else {}
    metrics_path = run_dir / "training_metrics.csv"
    histories[task] = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
    status_rows.append(
        {
            "task": task,
            "status": manifest.get("status", "not_finished"),
            "latest_epoch": latest.get("epoch"),
            "best_epoch": manifest.get("best_epoch"),
            "global_step": manifest.get("global_step"),
        }
    )
pd.DataFrame(status_rows)

In [ ]:
fig, axes = plt.subplots(len(TASKS), 1, figsize=(12, 3 * len(TASKS)), sharex=False)
for axis, task in zip(axes, TASKS, strict=True):
    frame = histories[task]
    if frame.empty:
        axis.text(0.5, 0.5, "No committed epochs", ha="center", va="center")
    else:
        axis.plot(frame["epoch"], frame["reward_mean"], alpha=0.25, label="epoch return")
        rolling = frame["reward_mean"].rolling(30, min_periods=1).mean()
        axis.plot(frame["epoch"], rolling, linewidth=2, label="rolling-30")
    axis.set_title(task)
    axis.set_ylabel("Return (higher is better)")
    axis.grid(alpha=0.2)
    axis.legend(loc="best")
axes[-1].set_xlabel("Committed epoch")
fig.tight_layout()
plt.show()